In [25]:
# must match the hist_set_method the solves use, so the threshold below is
# evaluated at the n_ell the stability constraints actually enforce
HIST_SET_METHOD = "instance_farmers"
from stable_platform_matchings.domain.hist_sets import status_quo_quantities_for_method

from __future__ import annotations

import pickle
import platform
import sys
from pathlib import Path
from typing import Any

import numpy as np

from stable_platform_matchings import Optimizer
from stable_platform_matchings.optimization.options import OptimizerParams, SolverOptions
from stable_platform_matchings.domain.instance import Instance
from stable_platform_matchings.graphs.road_graphs import RoadGraph
import stable_platform_matchings.experiments.utils as utils


BASE_SEED = 20260918
VRP_TIME_LIMIT_SECONDS = 900

N_SAMPLES = 100
EPSILON_ELL = 0.5
TOP_N = [1, 2, 3]

B = 14

def set_epsilons(
    instance: Instance,
    treatment_ids: set,
    epsilon_h: float,
    epsilon_ell: float,
) -> dict[str, float]:
    """Sample an epsilon for every intermediary."""
    epsilons = {}
    for intermediary in instance.intermediaries:
        if intermediary.id in treatment_ids:
            epsilons[intermediary.id] = epsilon_h
        else:
            epsilons[intermediary.id] = epsilon_ell
    return epsilons


def set_het_costs(
    instance: Instance,
    treatment_ids: set,
    margin: float = 1
) -> dict[str, float]:
    """Sample heterogeneous costs for every intermediary."""
    base_het_costs = {
        intermediary.id: float(2.0 * instance.dist_to_mill[intermediary.id])
        for intermediary in instance.intermediaries
    }

    # get min treatment and max control sigmas
    min_treatment_sigma = min(
        het_cost for intermediary_id, het_cost in base_het_costs.items()
        if intermediary_id in treatment_ids
    )
    max_control_sigma = max(
        het_cost for intermediary_id, het_cost in base_het_costs.items()
        if intermediary_id not in treatment_ids
    )

    # scale treatment sigmas
    scale_factor = max(1, (max_control_sigma + margin) / min_treatment_sigma)

    het_costs = {}
    for intermediary_id in base_het_costs:
        if intermediary_id in treatment_ids:
            het_costs[intermediary_id] = base_het_costs[intermediary_id] * scale_factor
        else:
            het_costs[intermediary_id] = base_het_costs[intermediary_id]

    return het_costs

def clip_het_costs(
    instance: Instance,
    treatment_ids: set,
    margin: float = 100
) -> dict[str, float]:
    """Sample heterogeneous costs for every intermediary."""
    base_het_costs = {
        intermediary.id: float(2.0 * instance.dist_to_mill[intermediary.id])
        for intermediary in instance.intermediaries
    }

    # get min treatment and max control sigmas
    max_control_sigma = max(
        het_cost for intermediary_id, het_cost in base_het_costs.items()
        if intermediary_id not in treatment_ids
    )

    het_costs = {}
    for intermediary_id in base_het_costs:
        if intermediary_id in treatment_ids:
            het_costs[intermediary_id] = max(base_het_costs[intermediary_id], max_control_sigma + margin)
        else:
            het_costs[intermediary_id] = base_het_costs[intermediary_id]

    return het_costs

In [26]:

# get data paths
data_path = Path("../data")
instances_path = data_path / "anon_14_day_instances"
graph_path = data_path / "graph_0-14960_00_new.pickle"

# check if data is well-formed
if not instances_path.is_dir():
    raise FileNotFoundError(f"Instance directory does not exist: {instances_path}")
if not graph_path.is_file():
    raise FileNotFoundError(f"Graph file does not exist: {graph_path}")

# load instance paths
instance_paths = sorted(
    path for path in instances_path.iterdir()
    if path.is_file()
    and path.suffix.lower() in {".yaml", ".yml"}
    and not path.name.startswith("aggregate")
)
if not instance_paths:
    raise FileNotFoundError(f"No YAML instance files found in {instances_path}")

with graph_path.open("rb") as file:
    graph = pickle.load(file)

holds = []
checks=[]

for b in range(14):
    job_id = b
    rng = np.random.default_rng(seed=int(job_id))
    epsilon_hs = rng.uniform(0, 9, size=N_SAMPLES)
    top_n_idx = np.random.randint(0, 3)

    # load instance
    print("Loading instance...")
    instance_path = instance_paths[b]
    instance = Instance.from_yaml(instance_path)

    print(f"Loaded instance {instance_path}, setting graph...")
    instance.set_graph(RoadGraph(graph))

    top_n = TOP_N[top_n_idx]
    status_quo_sorted = sorted(
        status_quo_quantities_for_method(instance, HIST_SET_METHOD).items(),
        key=lambda x: x[1], 
        reverse=True
    )
    treatment_ids = {item[0] for item in status_quo_sorted[:top_n]}

    check = False

    for epsilon_h in epsilon_hs:

        sigmas = set_het_costs(instance, treatment_ids)

        high_types = {
            intermediary_id: sigmas[intermediary_id]
            for intermediary_id in treatment_ids
        }

        low_types = {
            intermediary.id for intermediary in instance.intermediaries
            if intermediary.id not in treatment_ids
        }

        r_sigma_h = np.random.choice(list(high_types.values()))

        r_ell = np.random.choice(list(low_types))
        r_sigma_ell, r_n_ell = sigmas[r_ell], status_quo_quantities_for_method(instance, HIST_SET_METHOD)[r_ell]

        rhs = (r_sigma_h / r_sigma_ell * (r_n_ell + EPSILON_ELL))

        if epsilon_h > rhs:
            check = True

        holds.append(rhs)
    checks.append(check)

np.mean(checks)

Loading instance...
Loaded instance ../data/anon_14_day_instances/2020-08-27.yaml, setting graph...
Loading instance...
Loaded instance ../data/anon_14_day_instances/2020-08-28.yaml, setting graph...
Loading instance...
Loaded instance ../data/anon_14_day_instances/2020-08-29.yaml, setting graph...
Loading instance...
Loaded instance ../data/anon_14_day_instances/2020-08-30.yaml, setting graph...
Loading instance...
Loaded instance ../data/anon_14_day_instances/2020-08-31.yaml, setting graph...
Loading instance...
Loaded instance ../data/anon_14_day_instances/2020-09-01.yaml, setting graph...
Loading instance...
Loaded instance ../data/anon_14_day_instances/2020-09-02.yaml, setting graph...
Loading instance...
Loaded instance ../data/anon_14_day_instances/2020-09-03.yaml, setting graph...
Loading instance...
Loaded instance ../data/anon_14_day_instances/2020-09-04.yaml, setting graph...
Loading instance...
Loaded instance ../data/anon_14_day_instances/2020-09-05.yaml, setting graph...


np.float64(1.0)

In [27]:

# get data paths
data_path = Path("../data")
instances_path = data_path / "anon_14_day_instances"
graph_path = data_path / "graph_0-14960_00_new.pickle"

# check if data is well-formed
if not instances_path.is_dir():
    raise FileNotFoundError(f"Instance directory does not exist: {instances_path}")
if not graph_path.is_file():
    raise FileNotFoundError(f"Graph file does not exist: {graph_path}")

# load instance paths
instance_paths = sorted(
    path for path in instances_path.iterdir()
    if path.is_file()
    and path.suffix.lower() in {".yaml", ".yml"}
    and not path.name.startswith("aggregate")
)
if not instance_paths:
    raise FileNotFoundError(f"No YAML instance files found in {instances_path}")

with graph_path.open("rb") as file:
    graph = pickle.load(file)

holds = []
checks=[]

for b in range(14):
    job_id = b
    rng = np.random.default_rng(seed=int(job_id))
    epsilon_hs = rng.uniform(0, 9, size=N_SAMPLES)

    top_n_idx = np.random.randint(0, 3)

    # load instance
    print("Loading instance...")
    instance_path = instance_paths[b]
    instance = Instance.from_yaml(instance_path)

    print(f"Loaded instance {instance_path}, setting graph...")
    instance.set_graph(RoadGraph(graph))

    top_n = TOP_N[top_n_idx]
    status_quo_sorted = sorted(
        status_quo_quantities_for_method(instance, HIST_SET_METHOD).items(),
        key=lambda x: x[1], 
        reverse=True
    )
    treatment_ids = {item[0] for item in status_quo_sorted[:top_n]}

    check = False

    for epsilon_h in epsilon_hs:

        sigmas = clip_het_costs(instance, treatment_ids)

        high_types = {
            intermediary_id: sigmas[intermediary_id]
            for intermediary_id in treatment_ids
        }

        low_types = {
            intermediary.id for intermediary in instance.intermediaries
            if intermediary.id not in treatment_ids
        }

        r_sigma_h = np.random.choice(list(high_types.values()))

        r_ell = np.random.choice(list(low_types))
        r_sigma_ell, r_n_ell = sigmas[r_ell], status_quo_quantities_for_method(instance, HIST_SET_METHOD)[r_ell]

        rhs = (r_sigma_h / r_sigma_ell * (r_n_ell + EPSILON_ELL))

        if epsilon_h > rhs:
            check = True

        holds.append(rhs)
    checks.append(check)

np.mean(checks)

Loading instance...
Loaded instance ../data/anon_14_day_instances/2020-08-27.yaml, setting graph...
Loading instance...
Loaded instance ../data/anon_14_day_instances/2020-08-28.yaml, setting graph...
Loading instance...
Loaded instance ../data/anon_14_day_instances/2020-08-29.yaml, setting graph...
Loading instance...
Loaded instance ../data/anon_14_day_instances/2020-08-30.yaml, setting graph...
Loading instance...
Loaded instance ../data/anon_14_day_instances/2020-08-31.yaml, setting graph...
Loading instance...
Loaded instance ../data/anon_14_day_instances/2020-09-01.yaml, setting graph...
Loading instance...
Loaded instance ../data/anon_14_day_instances/2020-09-02.yaml, setting graph...
Loading instance...
Loaded instance ../data/anon_14_day_instances/2020-09-03.yaml, setting graph...
Loading instance...
Loaded instance ../data/anon_14_day_instances/2020-09-04.yaml, setting graph...
Loading instance...
Loaded instance ../data/anon_14_day_instances/2020-09-05.yaml, setting graph...


np.float64(1.0)